<a href="https://colab.research.google.com/github/MarkVanDyk/Reverse-Universe-Theory-R.U.T/blob/main/Simulation%20rut%20frozen%20vs%20rut%20dynamic%20from%20orbiting%20rippling%20spacetime%20vs%20LCMD%20%26%20Dark%20energy%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1A: minimal installs (run once)
!pip install numpy scipy matplotlib

# Step 1B: rut_phase prototype and mock theory
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Mock H(z) (flat LCDM) in units where H0=1 for simplicity
def H_of_z_mock(z, H0=1.0, Om=0.3):
    return H0 * np.sqrt(Om*(1+z)**3 + (1-Om))

def z_to_N(z):
    return -np.log(1.0 + z)

def N_to_z(N):
    return np.exp(-N) - 1.0

# ODE in e-fold N: phi'' + (3 + dlnH/dN + Gamma/H) phi' + (mu^2/H^2)(phi-phi0) = 0
def integrate_phi_over_z(z_array, mu, Gamma, phi0, phi_init=0.0, phiN_init=0.0):
    N_array = z_to_N(np.array(z_array))
    # build H(N) from H(z)
    z_for_H = np.unique(np.clip(z_array, 0.0, np.max(z_array)))
    N_for_H = z_to_N(z_for_H)
    H_for_H = np.array([H_of_z_mock(z) for z in z_for_H])
    def H_of_N(N):
        return np.interp(N, N_for_H, H_for_H, left=H_for_H[0], right=H_for_H[-1])
    def rhs(N, y):
        phi, u = y
        H = H_of_N(N)
        # finite-diff dlnH/dN
        dN = 1e-5
        Hp = H_of_N(N + dN)
        dlnH_dN = (Hp - H)/ (H * dN + 1e-30)
        Gamma_N = Gamma / (H + 1e-30)
        phiNN = - (3.0 + dlnH_dN + Gamma_N) * u - (mu**2 / (H**2 + 1e-30)) * (phi - phi0)
        return [u, phiNN]
    y0 = [phi_init, phiN_init]
    sol = solve_ivp(rhs, (N_array.min(), N_array.max()), y0, t_eval=N_array, rtol=1e-8, atol=1e-10)
    return sol.y[0]  # phi(N) sampled at N_array

# Step 1C: compute f(z) and plot
z_test = np.linspace(0, 3, 200)  # redshift range to inspect
params = dict(mu=0.1, Gamma=0.05, phi0=0.5, b_C=0.1, b_S=0.05)
phi_vals = integrate_phi_over_z(z_test, params['mu'], params['Gamma'], params['phi0'])
f_z = 1.0 + params['b_C']*np.cos(phi_vals) + params['b_S']*np.sin(phi_vals)

plt.figure(figsize=(8,4))
plt.plot(z_test, f_z, label='f(z)')
plt.axhline(1.0, color='k', lw=0.6, ls='--')
plt.xlabel('z'); plt.ylabel('f(z)'); plt.title('RUT modulation prototype')
plt.legend(); plt.grid(True)
plt.show()

# Step 1D: quick sanity checks
print("f(z) min, max:", f_z.min(), f_z.max())


ValueError: Values in `t_eval` are not properly sorted.